In [48]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
data_dir = Path('/content/drive/MyDrive/IDXExchange_Training/data')

import pandas as pd
import numpy as np
from pathlib import Path
# Run in Anaconda Prompt or terminal if needed:
# pip install pandas numpy scikit-learn xgboost matplotlib joblib

# Confirming files exist
from pathlib import Path
data_dir = Path('/content/drive/MyDrive/IDXExchange_Training/data')
training_files = [
 'CRMLSSold202509.csv', 'CRMLSSold202510.csv',
 'CRMLSSold202511.csv', 'CRMLSSold202512.csv',
 'CRMLSSold202601.csv', 'CRMLSSold202602.csv',
]
test_file = 'CRMLSSold202603.csv'
for name in training_files + [test_file]:
 path = data_dir / name
 print(name, 'exists:', path.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CRMLSSold202509.csv exists: True
CRMLSSold202510.csv exists: True
CRMLSSold202511.csv exists: True
CRMLSSold202512.csv exists: True
CRMLSSold202601.csv exists: True
CRMLSSold202602.csv exists: True
CRMLSSold202603.csv exists: True


## Week 1

In [49]:
import pandas as pd
from pathlib import Path
data_dir = Path('/content/drive/MyDrive/IDXExchange_Training/data')
training_files = [
 data_dir / 'CRMLSSold202509.csv',
 data_dir / 'CRMLSSold202510.csv',
 data_dir / 'CRMLSSold202511.csv',
 data_dir / 'CRMLSSold202512.csv',
 data_dir / 'CRMLSSold202601.csv',
 data_dir / 'CRMLSSold202602.csv',
]
dfs = [pd.read_csv(f) for f in training_files]
df = pd.concat(dfs, ignore_index=True)
print('Combined training shape:', df.shape)

df = df[
 (df['PropertyType'] == 'Residential') &
 (df['PropertySubType'] == 'SingleFamilyResidence')
].copy()

numeric_cols = [
 'ClosePrice', 'BedroomsTotal', 'BathroomsTotalInteger', 'LivingArea',
 'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces', 'Stories',
 'Latitude', 'Longitude'
]
for col in numeric_cols:
 if col in df.columns:
  df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df['ClosePrice'] > 0].copy()
df = df[df['LivingArea'] > 0].copy()
df = df.dropna(subset=['ClosePrice', 'BedroomsTotal',
 'BathroomsTotalInteger', 'LivingArea', 'PostalCode']).copy()

/tmp/ipykernel_791/4243774183.py:12: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(f) for f in training_files]


Combined training shape: (119913, 78)


## Week 2

In [50]:
# Sentinel fill: null GarageSpaces means no garage
df['GarageSpaces'] = df['GarageSpaces'].fillna(0)
# Median fill for columns with modest null rates
for col in ['LotSizeSquareFeet', 'YearBuilt', 'Stories', 'Latitude', 'Longitude']:
 if col in df.columns:
  df[col] = df[col].fillna(df[col].median())


df['PostalCode5'] = df['PostalCode'].astype(str).str.slice(0, 5)
df['City'] = df['City'].astype(str)
# These use the full training DataFrame -- do NOT include test data here
#df['zip_median_price'] = df.groupby('PostalCode5')['ClosePrice'].transform('median')
#df['city_median_price'] = df.groupby('City')['ClosePrice'].transform('median')


df_constrained = df[df['ClosePrice'] < 5_000_000].copy()

## Week 3

In [51]:
from sklearn.model_selection import train_test_split

# Features that do NOT depend on ClosePrice
base_features = [
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'LivingArea',
    'LotSizeSquareFeet',
    'YearBuilt',
    'GarageSpaces',
    'Stories',
    'Latitude',
    'Longitude',
    'PostalCode5',
    'City'
]

model_df = df_constrained[base_features + ['ClosePrice']].dropna().copy()

# Split BEFORE calculating ZIP/city median prices
train_df, val_df = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42
)

# Calculate medians using TRAINING data only
zip_map = train_df.groupby('PostalCode5')['ClosePrice'].median()
city_map = train_df.groupby('City')['ClosePrice'].median()

# Use the overall training median as a fallback
global_median = train_df['ClosePrice'].median()

# Create ZIP/city median features for training
train_df['zip_median_price'] = (
    train_df['PostalCode5']
    .map(zip_map)
    .fillna(global_median)
)

train_df['city_median_price'] = (
    train_df['City']
    .map(city_map)
    .fillna(global_median)
)

# Create ZIP/city median features for validation
val_df['zip_median_price'] = (
    val_df['PostalCode5']
    .map(zip_map)
    .fillna(global_median)
)

val_df['city_median_price'] = (
    val_df['City']
    .map(city_map)
    .fillna(global_median)
)


In [52]:
feature_cols = [
 'BedroomsTotal', 'BathroomsTotalInteger', 'LivingArea', 'LotSizeSquareFeet',
 'YearBuilt', 'GarageSpaces', 'Stories', 'Latitude', 'Longitude',
 'zip_median_price', 'city_median_price'
]
X_train = train_df[feature_cols]
y_train = train_df['ClosePrice']

X_val = val_df[feature_cols]
y_val = val_df['ClosePrice']

In [53]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error
def mdape(y_true, y_pred):
 y_true = np.array(y_true)
 y_pred = np.array(y_pred)
 mask = y_true != 0
 ape = np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]) * 100
 return np.median(ape)

def regression_report(y_true, y_pred):
  return {
  'R2': r2_score(y_true, y_pred),
  'MAE': mean_absolute_error(y_true, y_pred),
  'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
  'MedianAbsoluteError': median_absolute_error(y_true, y_pred),
  'MDAPE': mdape(y_true, y_pred),
  }

Train Models

In [54]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
lr = LinearRegression()
lr.fit(X_train, y_train)
print('Linear Regression:', regression_report(y_val, lr.predict(X_val)))

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print('Random Forest:', regression_report(y_val, rf.predict(X_val)))

Linear Regression: {'R2': 0.8008849852676533, 'MAE': 211444.62835779833, 'RMSE': np.float64(335965.21231545287), 'MedianAbsoluteError': np.float64(133189.23901065218), 'MDAPE': np.float64(15.275820589115737)}
Random Forest: {'R2': 0.8755869358937715, 'MAE': 148754.66448485266, 'RMSE': np.float64(265567.72823092656), 'MedianAbsoluteError': np.float64(72517.59999999998), 'MDAPE': np.float64(8.470247354838712)}


Cross Validation

In [55]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(lr, X, y, cv=5, scoring='neg_mean_absolute_error')
print('CV MAE (mean):', -cv_scores.mean())
print('CV MAE (std): ', cv_scores.std())
# A large std means performance varies a lot across folds -- investigate why

CV MAE (mean): 471915.20736866165
CV MAE (std):  86823.6054069214


## Deliverable

Metric Summary for Linear Regression & Random Forest

In [56]:
lr_metrics = regression_report(y_val, lr.predict(X_val))
rf_metrics = regression_report(y_val, rf.predict(X_val))

pd.DataFrame({
    'Metric': list(lr_metrics.keys()),
    'Linear Regression': list(lr_metrics.values()),
    'Random Forest': list(rf_metrics.values())
}).set_index('Metric').round(2)

,Linear Regression,Random Forest
Metric,,
R2,0.80,0.88
MAE,211444.63,148754.66
RMSE,335965.21,265567.73
MedianAbsoluteError,133189.24,72517.60
MDAPE,15.28,8.47


Model Comparison:
- Stronger Model: Random Forest
  - Higher R², explaining more of the variation in property sale prices
  - Lower MAE, on average the predictions were closer to the actual price
  - Lower RMSE, fewer prediction errors
  - Lower Median absolute error, representing a lower typical prediction error
  - Lower MdAPE, percentage error is lower

Although Random Forest performed better, the model still has prediction error for some properties with larger prices.